# Проверка на своих картинках

Модель — ансамбль экспертов на общем стволе. Общий эксперт `all` обучен на всех генераторах, остальные — каждый на
своём семействе: OpenAI, Google, Flux, Qwen и так далее. Эксперт отвечает, сгенерирована ли картинка его
семейством, и называет самую похожую модель. Картинка считается сгенерированной, если хотя бы один эксперт выше
своего порога. Пороги пишет `scripts/run_test.py` в `thresholds.json`: все эксперты вместе ложно срабатывают
примерно на 1 % реальных картинок. Без этого файла пороги 0.5.

Ствол считается один раз на картинку, поэтому всё работает и на CPU: около половины секунды на картинку.

In [ ]:
import os, io, json, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

from pathlib import Path
import numpy as np
import yaml, torch
import matplotlib.pyplot as plt
from PIL import Image
from src.model import Detector, score

cfg = yaml.safe_load(open('configs/train.yaml'))
out = Path(cfg['train']['out'])
size = cfg['train']['size']   # 512
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = Detector.load(out, device)
names = list(model.experts)

# пороги экспертов из теста; без него 0.5
thresholds = {name: 0.5 for name in names}
if (out / 'thresholds.json').exists():
    thresholds = json.load(open(out / 'thresholds.json'))
thresholds

Картинка готовится как на тесте: не-JPEG один раз сжимается в JPEG с качеством 90 (на обучении так сжимались
все не-JPEG), потом картинка целиком сжимается до 512×512.

In [ ]:
def prepare(path):
    image = Image.open(path)
    fmt, image = image.format, image.convert('RGB')
    if fmt != 'JPEG':
        buf = io.BytesIO()
        image.save(buf, 'JPEG', quality=90)
        image = Image.open(buf).convert('RGB')
    image = image.resize((size, size), Image.BICUBIC)
    return torch.from_numpy(np.array(image, dtype=np.float32) / 255).permute(2, 0, 1)   # [3, 512, 512], от 0 до 1


@torch.no_grad()
def predict(path):
    x = prepare(path)[None].to(device)
    with torch.autocast('cuda', dtype=torch.bfloat16, enabled=device == 'cuda'):
        logits = model(x)
    result = {}
    for name in names:
        prob, index = score(logits[name])
        result[name] = (prob.item(), model.experts[name].models[index.item()])
    return result   # эксперт -> (вероятность, что картинка сгенерирована; самая похожая модель эксперта)

In [ ]:
path = 'my_images/example.jpg'   # своя картинка

result = predict(path)
fired = [n for n in names if result[n][0] > thresholds[n]]              # эксперты выше своего порога
if fired:
    expert = max(fired, key=lambda n: result[n][0] - thresholds[n])     # кто дальше всех от порога
    verdict = f'сгенерирована: эксперт {expert}, похоже на {result[expert][1]}'
else:
    verdict = 'реальная'
print(verdict)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].imshow(Image.open(path).convert('RGB'))
ax[0].set_title(verdict)
ax[0].axis('off')
ax[1].barh(names, [result[n][0] for n in names], color=['tab:red' if n in fired else 'tab:gray' for n in names])
for i, n in enumerate(names):
    ax[1].plot([thresholds[n]] * 2, [i - 0.4, i + 0.4], color='black')   # порог эксперта
    ax[1].text(1.02, i, result[n][1], va='center', fontsize=9)           # самая похожая модель
ax[1].set_xlim(0, 1)
ax[1].invert_yaxis()
ax[1].set_title('вероятность по экспертам, чёрная черта — порог')
plt.tight_layout()
plt.show()

In [ ]:
# все картинки из папки
folder = Path('my_images')   # папка, не файл
for p in sorted(folder.iterdir()):
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}:
        result = predict(p)
        fired = [n for n in names if result[n][0] > thresholds[n]]
        expert = max(fired, key=lambda n: result[n][0] - thresholds[n]) if fired else ''
        print(f"{'FAKE' if fired else 'real'}  {expert:10s}  {result[expert][1] if fired else '':32s}  {p.name}")